# Football Scouting Dashboard

## Notebook 02 - Feature Selection

### Objective

Select the most meaningful features for building a football scouting model.

The goal is not to use every available statistic.

The goal is to identify the football attributes that best represent a player's style and ability.

### Goals

- Understand different types of dataset columns
- Separate metadata from model features
- Remove duplicate statistics
- Remove derived statistics that create double counting
- Categorize features into football attributes
- Create the final feature blueprint for preprocessing
- Decide which features should influence player similarity

### Output

A feature selection blueprint containing:

- `ATTRIBUTE_MAP`
- `FEATURE_COLUMNS`
- `METADATA_COLUMNS`
- `COUNT_COLUMNS`
- `PERCENTAGE_COLUMNS`

This notebook will not modify the dataset.

It only defines what information the scouting model should learn from.

In [26]:
import pandas as pd

df = pd.read_csv("../data/raw/players_data_light-2024_2025.csv")

df.shape

(2854, 165)

In [27]:
METADATA_COLUMNS = [
    "Player",
    "Nation",
    "Pos",
    "Squad",
    "Comp",
    "Age",
    "Born",
    "Min"
]

In [28]:
DUPLICATE_COLUMNS = [
    "Ast_stats_passing",
    "xAG_stats_passing",
    "PrgP_stats_passing",
    "PrgC_stats_possession",
    "Int_stats_misc",
    "TklW_stats_misc",
    "CrdY_stats_misc",
    "CrdR_stats_misc",
]

In [29]:
TEMPORARY_COLUMNS = [
    "MP",
    "Starts",
    "Min",
    "90s"
]

In [30]:
ATTRIBUTE_MAP = {

    "Finishing": {
        "description": "Goal scoring and shot quality",
        "features": [
            "Gls",
            "xG",
            "Sh",
            "SoT",
            "G/Sh"
        ],
        "aggregation": "mean"
    },

    "Creativity": {
        "description": "Chance creation and attacking playmaking",
        "features": [
            "Ast",
            "xA",
            "KP",
            "SCA",
            "GCA",
            "TB"
        ],
        "aggregation": "mean"
    },

    "Passing": {
        "description": "Progressive and accurate passing ability",
        "features": [
            "Cmp%",
            "PrgP",
            "PrgDist",
            "PPA",
            "TB",
            "Sw"
        ],
        "aggregation": "mean"
    },

    "Progression": {
        "description": "Advancing the ball through carries and passes",
        "features": [
            "PrgC",
            "PrgP",
            "Carries",
            "CPA",
            "Succ"
        ],
        "aggregation": "mean"
    },

    "Defending": {
        "description": "Ball-winning and defensive actions",
        "features": [
            "Tkl",
            "TklW",
            "Int",
            "Blocks_stats_defense",
            "Clr",
            "Recov"
        ],
        "aggregation": "mean"
    },

    "BallSecurity": {
        "description": "Ability to retain possession and avoid turnovers",
        "features": [
            "Cmp%",
            "Mis",
            "Dis",
            "Rec"
        ],
        "aggregation": "mean"
    },

    "Aerial": {
        "description": "Effectiveness in aerial duels",
        "features": [
            "Won",
            "Won%"
        ],
        "aggregation": "mean"
    }

}

In [31]:
DISCIPLINE_COLUMNS = [
    "CrdY",
    "CrdR",
    "2CrdY",
    "Fls"
]

In [32]:

DERIVED_COLUMNS = [
    "G+A",
    "G-PK",
    "G+A-PK",
    "xG+xAG",
    "npxG+xAG",
    "G-xG",
    "np:G-xG",
]

In [33]:
POSITION_GROUPS = {

    "DF": [
        "DF",
        "DF,MF",
        "MF,DF",
        "DF,FW",
        "FW,DF"
    ],

    "MF": [
        "MF",
        "MF,FW",
        "FW,MF"
    ],

    "FW": [
        "FW"
    ]

}

In [34]:
CLUSTER_FEATURES = []

for attribute, columns in ATTRIBUTE_MAP.items():
    CLUSTER_FEATURES.extend(columns)


CLUSTER_FEATURES = sorted(set(CLUSTER_FEATURES))

len(CLUSTER_FEATURES)

3

In [35]:
missing_features = [
    col for col in CLUSTER_FEATURES
    if col not in df.columns
]

missing_features

['aggregation', 'description', 'features']

In [36]:
PREPROCESSING_CONFIG = {
    "min_minutes": 600,
    "remove_goalkeepers": True,
    "merge_transfers": True,
    "primary_position_only": True
}

In [38]:
import json
from pathlib import Path

config_dir = Path("../config")
config_dir.mkdir(exist_ok=True)

with open(config_dir / "feature_config.json", "w") as f:
    json.dump(
        {
            "metadata_columns": METADATA_COLUMNS,
            "duplicate_columns": DUPLICATE_COLUMNS,
            "derived_columns": DERIVED_COLUMNS,
            "temporary_columns": TEMPORARY_COLUMNS,
            "discipline_columns": DISCIPLINE_COLUMNS,
            "position_groups": POSITION_GROUPS,
            "attribute_map": ATTRIBUTE_MAP,
        },
        f,
        indent=4,
    )

with open(config_dir / "preprocessing_config.json", "w") as f:
    json.dump(PREPROCESSING_CONFIG, f, indent=4)